# NB12 — Contributing: Add a New Model

**North star callback:** Every optimization in NB02–NB10 lives in a model-specific patch
file. When a new architecture lands in HuggingFace (Qwen3, Gemma2, …), Unsloth adds it
by creating one new file and touching two existing ones. This notebook is a guided tour
and copy-pasteable template for doing exactly that.

## 1. Codebase map

In [1]:
from pathlib import Path

root = Path('../unsloth/unsloth')

sections = [
    ('Entry points',       ['__init__.py', 'models/__init__.py', 'models/loader.py']),
    ('Per-arch patches',   ['models/llama.py', 'models/mistral.py', 'models/gemma.py',
                            'models/qwen2.py', 'models/qwen3.py', 'models/_utils.py']),
    ('Triton kernels',     ['kernels/fast_lora.py', 'kernels/cross_entropy_loss.py',
                            'kernels/rope_embedding.py', 'kernels/utils.py']),
]

for section, files in sections:
    print(f'\n{section}')
    print('─' * 60)
    for fname in files:
        path = root / fname
        if path.exists():
            lines = sum(1 for _ in open(path))
            print(f'  unsloth/{fname:45s} {lines:5d} lines')
        else:
            print(f'  unsloth/{fname:45s}  (not found)')


Entry points
────────────────────────────────────────────────────────────
  unsloth/__init__.py                                     353 lines
  unsloth/models/__init__.py                               31 lines
  unsloth/models/loader.py                               1643 lines

Per-arch patches
────────────────────────────────────────────────────────────
  unsloth/models/llama.py                                3614 lines
  unsloth/models/mistral.py                               487 lines
  unsloth/models/gemma.py                                 493 lines
  unsloth/models/qwen2.py                                 101 lines
  unsloth/models/qwen3.py                                 478 lines
  unsloth/models/_utils.py                               3251 lines

Triton kernels
────────────────────────────────────────────────────────────
  unsloth/kernels/fast_lora.py                            730 lines
  unsloth/kernels/cross_entropy_loss.py                   463 lines
  unsloth/kernels/rop

## 2. Anatomy of a model patch file

Each `models/<arch>.py` follows the same structure:
1. **Fast forward functions** — free functions that replace HF class methods
2. **`FastXxxModel` class** — holds `pre_patch()`, `post_patch()`, `from_pretrained()`
3. `pre_patch()` wires the fast forwards onto HF classes via `setattr` (NB09 pattern)
4. `post_patch()` fixes tokenizer quirks and weight-level issues after loading

In [2]:
import sys, inspect
sys.path.insert(0, '../unsloth')

from unsloth.models.llama import FastLlamaModel

# List all methods / staticmethods on FastLlamaModel
print('FastLlamaModel interface:')
for name in dir(FastLlamaModel):
    if name.startswith('_'):
        continue
    obj = getattr(FastLlamaModel, name)
    kind = 'staticmethod' if isinstance(inspect.getattr_static(FastLlamaModel, name), staticmethod) else type(obj).__name__
    print(f'  {name:35s}  [{kind}]')

print()
print('pre_patch source (the 7 setattr calls seen in NB09):')
src = inspect.getsource(FastLlamaModel.pre_patch)
# Show only the setattr lines
for line in src.splitlines():
    if '.forward' in line or 'def pre_patch' in line or 'fix_prepare' in line:
        print(' ', line)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
FastLlamaModel interface:
  for_inference                        [staticmethod]
  for_training                         [staticmethod]
  from_pretrained                      [staticmethod]
  get_peft_model                       [staticmethod]
  patch_peft_model                     [staticmethod]
  post_patch                           [staticmethod]
  pre_patch                            [staticmethod]

pre_patch source (the 7 setattr calls seen in NB09):
      def pre_patch():
          LlamaAttention.forward = LlamaAttention_fast_forward
          LlamaSdpaAttention.forward = LlamaAttention_fast_forward
          LlamaFlashAttention2.forward = LlamaAttention_fast_forward
          LlamaDecoderLayer.forward = LlamaDecoderLayer_

## 3. The three registration points

Adding a new architecture requires exactly three edits:

| File | Change |
|---|---|
| `models/my_model.py` | **Create** — `FastMyModel` class with `pre_patch` / `post_patch` |
| `models/__init__.py` | **Add** one import line |
| `models/loader.py`   | **Add** one `elif model_type == "my_model":` branch |

In [3]:
# Show the relevant section of loader.py dispatch table
loader_path = Path('../unsloth/unsloth/models/loader.py')
lines = loader_path.read_text().splitlines()

# Find the dispatch block
start = next(i for i, l in enumerate(lines) if 'model_type == "llama"' in l)
end   = next(i for i, l in enumerate(lines) if i > start and 'dispatch_model =' in l and 'FastQwen' in l) + 1

print('loader.py — dispatch block (excerpt):')
print()
for lineno, line in enumerate(lines[start-1:end+4], start=start):
    print(f'{lineno:5d}  {line}')

print()
print('# To add MyNewModel, append after the last elif:')
print('        elif model_type == "my_new_model":')
print('            dispatch_model = FastMyNewModel')

print()
# Show __init__.py imports
init_path = Path('../unsloth/unsloth/models/__init__.py')
print('models/__init__.py — current imports:')
for line in init_path.read_text().splitlines()[:20]:
    if line.strip():
        print(' ', line)

print()
print('# Add:')
print('from .my_new_model import FastMyNewModel')

loader.py — dispatch block (excerpt):

  578  
  579          if model_type == "llama":
  580              scaling_type = None
  581              if getattr(model_config, "rope_scaling", None) is not None:
  582                  scaling_type1 = model_config.rope_scaling.get("type", None)
  583                  scaling_type2 = model_config.rope_scaling.get("rope_type", None)
  584                  scaling_type = (
  585                      scaling_type1 if scaling_type1 is not None else scaling_type2
  586                  )
  587  
  588              if scaling_type == "llama3" and not SUPPORTS_LLAMA31:
  589                  raise ImportError(
  590                      f"Unsloth: Your transformers version of {transformers_version} does not support Llama 3.1.\n"
  591                      f"The minimum required version is 4.43.2\n"
  592                      f'Try `pip install --upgrade "transformers>=4.43.2"`\n'
  593                      f"to obtain the latest transformers build, t

## 4. Minimal patch template

The cell below is a complete, runnable skeleton. Copy it to
`unsloth/models/my_new_model.py`, fill in the fast-forward functions,
and register it in `__init__.py` and `loader.py`.

In [4]:
PATCH_TEMPLATE = '''
# unsloth/models/my_new_model.py
# Copy this file for a new architecture.
# Minimum: implement pre_patch(), register in __init__.py and loader.py.

import torch
from unsloth.models._utils import patch_tokenizer


# -----------------------------------------------------------------------
# 1. Fast forward functions
#    These are plain functions — not methods — so they can be easily tested
#    in isolation and swapped without touching the HF class hierarchy.
# -----------------------------------------------------------------------

def MyNewModelAttention_fast_forward(self, hidden_states, attention_mask=None,
                                     position_ids=None, **kwargs):
    """
    Drop-in for MyNewModelAttention.forward.
    Suggested speedups (from earlier notebooks):
      - Fused RoPE kernel (NB05)                 → 3.4x for this op
      - F.scaled_dot_product_attention (NB04)    → 3.2x for this op
      - matmul_lora for QKV/O projections (NB07) → avoids HBM roundtrip
    """
    raise NotImplementedError


def MyNewModelMLP_fast_forward(self, hidden_states):
    """
    Drop-in for MyNewModelMLP.forward.
    Suggested speedup: LoRA_MLP fused kernel (NB07).
    """
    raise NotImplementedError


def MyNewModelDecoderLayer_fast_forward(self, hidden_states, **kwargs):
    """Thin wrapper that calls the fast attention + fast MLP."""
    raise NotImplementedError


# -----------------------------------------------------------------------
# 2. FastMyNewModel — the registry entry
# -----------------------------------------------------------------------

class FastMyNewModel:

    @staticmethod
    def pre_patch():
        """Called by loader.py BEFORE weights are loaded."""
        from transformers.models.my_new_model.modeling_my_new_model import (
            MyNewModelAttention,
            MyNewModelDecoderLayer,
        )
        MyNewModelAttention.forward    = MyNewModelAttention_fast_forward
        MyNewModelDecoderLayer.forward = MyNewModelDecoderLayer_fast_forward
        # Patch RoPE classes if the architecture uses a non-standard variant:
        # patch_llama_rope_scaling(model_name="my_new_model", ...)
        return

    @staticmethod
    def post_patch(model, tokenizer):
        """Called AFTER weights are loaded. Fix tokenizer quirks here."""
        model, tokenizer = patch_tokenizer(model, tokenizer)
        return model, tokenizer

    @staticmethod
    def from_pretrained(
        model_name,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
        **kwargs,
    ):
        """Convenience wrapper — delegates to FastLanguageModel.from_pretrained."""
        from unsloth.models.loader import FastLanguageModel
        return FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            dtype=dtype,
            load_in_4bit=load_in_4bit,
            model_patcher=FastMyNewModel,
            **kwargs,
        )
'''

print(PATCH_TEMPLATE)


# unsloth/models/my_new_model.py
# Copy this file for a new architecture.
# Minimum: implement pre_patch(), register in __init__.py and loader.py.

import torch
from unsloth.models._utils import patch_tokenizer


# -----------------------------------------------------------------------
# 1. Fast forward functions
#    These are plain functions — not methods — so they can be easily tested
#    in isolation and swapped without touching the HF class hierarchy.
# -----------------------------------------------------------------------

def MyNewModelAttention_fast_forward(self, hidden_states, attention_mask=None,
                                     position_ids=None, **kwargs):
    """
    Drop-in for MyNewModelAttention.forward.
    Suggested speedups (from earlier notebooks):
      - Fused RoPE kernel (NB05)                 → 3.4x for this op
      - F.scaled_dot_product_attention (NB04)    → 3.2x for this op
      - matmul_lora for QKV/O projections (NB07) → avoids HBM roundtrip
    ""

## 5. Smoke-test the template structure

In [5]:
import torch
import torch.nn as nn

# ---- Toy "HuggingFace model" to patch ----

class ToyAttention(nn.Module):
    def forward(self, x):
        return x  # trivial baseline

class ToyDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = ToyAttention()
    def forward(self, x):
        return self.attn(x)


# ---- Fast forward (the "optimization") ----

def fast_toy_attention(self, x):
    return x * 2.0  # pretend this is faster


# ---- Minimal FastToyModel following the template ----

class FastToyModel:
    @staticmethod
    def pre_patch():
        ToyAttention.forward = fast_toy_attention
        print('  pre_patch: ToyAttention.forward → fast_toy_attention')

    @staticmethod
    def post_patch(model, tokenizer=None):
        print('  post_patch: nothing to fix for ToyModel')
        return model, tokenizer


# ---- Simulate loader.py dispatch ----

print('Before patching:')
model = ToyDecoder()
x = torch.ones(1, 4)
print(f'  output = {model(x).tolist()}')

print('Applying patches (pre_patch):')
FastToyModel.pre_patch()
print(f'  output = {model(x).tolist()}')

print('Post-patch:')
FastToyModel.post_patch(model)

# Confirm patch is live on all instances
model2 = ToyDecoder()
print(f'\nNew instance output = {model2(x).tolist()}  (patch applies globally)')

Before patching:
  output = [[1.0, 1.0, 1.0, 1.0]]
Applying patches (pre_patch):
  pre_patch: ToyAttention.forward → fast_toy_attention
  output = [[2.0, 2.0, 2.0, 2.0]]
Post-patch:
  post_patch: nothing to fix for ToyModel

New instance output = [[2.0, 2.0, 2.0, 2.0]]  (patch applies globally)


## 6. Contribution workflow

```
# 1. Fork and clone
git clone https://github.com/<your-fork>/unsloth.git
cd unsloth
git checkout -b feat/add-my-new-model

# 2. Create the patch file
cp unsloth/models/llama.py unsloth/models/my_new_model.py
# Edit: replace LlamaXxx → MyNewModelXxx throughout

# 3. Register
# models/__init__.py  → add import
# models/loader.py    → add elif branch in dispatch block

# 4. Smoke-test (no real model needed)
python -c "
from unsloth.models.my_new_model import FastMyNewModel
FastMyNewModel.pre_patch()
print('pre_patch OK')
"

# 5. Integration test with a real checkpoint
python -c "
from unsloth import FastMyNewModel
model, tokenizer = FastMyNewModel.from_pretrained(
    'org/my-new-model-7b-bnb-4bit', max_seq_length=2048,
)
print(model)
"

# 6. Open PR against unsloth/unsloth main
#    Title: "[New model] MyNewModel support"
#    Include: architecture notes, any deviations from LLaMA pattern,
#             benchmark numbers (latency + VRAM) vs HF baseline
```

### Checklist before opening the PR

- [ ] `pre_patch()` leaves no un-reverted global state (or documents why it can't)
- [ ] `post_patch()` handles the tokenizer's `eos_token` / `pad_token` edge cases
- [ ] The fast attention forward passes `attention_mask` and `position_ids` through unchanged
- [ ] Runs without error with `load_in_4bit=True` (QLoRA path) and `load_in_4bit=False`
- [ ] Latency and VRAM numbers included in the PR description

## 7. Exercises

1. **Port the fused RoPE from NB05**: add `apply_rope_fused` (from NB05) to
   `MyNewModelAttention_fast_forward`. The HuggingFace signature is
   `apply_rotary_pos_emb(q, k, cos, sin, position_ids, unsqueeze_dim=1)` —
   adapt the wrapper to match, then verify correctness against the HF reference.

2. **Add a real architecture**: find a model on HuggingFace Hub whose
   `model_type` string is not yet in Unsloth's `loader.py` dispatch table.
   Create the patch file skeleton, get `pre_patch()` running on a CPU-only
   dummy model, and open a draft PR against your fork.

3. **Measure your patch**: once the patched model loads, run the `compare()`
   harness from `utils/benchmark.py` on one attention forward pass —
   HF baseline vs your patched version. Report the latency and VRAM delta.